# RtForecastR: A Walkthrough Tutorial

This notebook walks through real-time reproduction number ($R_t$) estimation and one-step-ahead forecasting, using the same code as the [RtForecastR](https://github.com/rajsubediresearch/RtForecastR) repository - just with explanations and interpretation alongside each step.

**Author:** Raj Subedi

**Attribution:** This tool builds on two published methods:
- Core $R_t$ estimation algorithm: Parag KV. (2021) "Improved estimation of time-varying reproduction numbers at low case incidence and between epidemic waves." *PLOS Computational Biology* 17(9): e1009347. [github.com/kpzoo/EpiFilter](https://github.com/kpzoo/EpiFilter)
- Generation-interval discretization convention: Chowell G, et al. (2024) "GrowthPredict..." *Scientific Reports* 14, 1630. [github.com/gchowell/forecasting_growthmodels](https://github.com/gchowell/forecasting_growthmodels)

The forecasting extension, plots, and this tutorial are original work by Raj Subedi.

---

**Before you start:** make sure this notebook's runtime is set to **R**. In Colab: `Runtime > Change runtime type > R`.


## Step 0: Setup

We install the one dependency (`caTools`) and pull the core algorithm files directly from the live GitHub repository, so this notebook always stays in sync with the actual codebase rather than a copy that can drift out of date.


In [ ]:
install.packages("caTools", quiet = TRUE)
suppressMessages(library(caTools))

dir.create("core", showWarnings = FALSE)
core_files <- c("epiFilter.R", "epiSmoother.R", "recursPredictExt.R", "computeLambda.R")
for (f in core_files) {
  download.file(
    paste0("https://raw.githubusercontent.com/rajsubediresearch/RtForecastR/main/core/", f),
    file.path("core", f), quiet = TRUE
  )
  source(file.path("core", f))
}
cat("Setup complete - core functions loaded.\n")

## Step 1: Load your data

Paste your own case-count time series into `data_text` below, replacing the example. Format: two columns, space-separated, one row per time point - `time_index case_count`, no header.

The example data here is a real measles outbreak (Ciudad de Mexico, 2025-2026 season).


In [ ]:
data_text <- "
16 5
17 1
18 11
19 8
20 13
21 11
22 14
23 27
24 41
25 40
26 65
27 85
28 85
29 61
30 82
31 63
32 77
33 46
34 49
35 54
36 46
37 46
38 22
39 25
40 18
41 16
42 11
43 3
44 6
45 3
46 8
47 3
"

data1 <- read.table(text = data_text, header = FALSE)
timevect <- data1[[1]]
I <- data1[[2]]
n <- length(I)

cat("Loaded", n, "time points, from t =", min(timevect), "to t =", max(timevect), "\n")
plot(timevect, I, type = "b", pch = 19, xlab = "Time", ylab = "Cases", main = "Observed case counts")

**Interpretation:** This is your raw input - nothing has been modeled yet. Look for the shape here before proceeding: a single clear peak (like this measles outbreak) is the easiest case for the methods below. Multiple peaks, very low counts, or long stretches of zeros can all still be analyzed, but the results are noisier and worth interpreting more cautiously in those cases.

## Step 2: Generation interval

$R_t$ estimation via the renewal equation requires the **generation interval** distribution - the time between when one person gets infected and when they infect someone else. We model it as a Gamma distribution, parameterized by its mean and standard deviation (converted to variance).

$$w_s = F_\Gamma(s) - F_\Gamma(s-1), \qquad s = 1, 2, \dots$$

where $F_\Gamma$ is the Gamma CDF with shape $k = \mu^2/\sigma^2$ and scale $\theta = \sigma^2/\mu$ (method-of-moments parameterization from mean $\mu$ and variance $\sigma^2$).

This discretized distribution $w_s$ is then convolved with past incidence to get the **total infectiousness** at each time point:

$$\Lambda_t = \sum_{s=1}^{t-1} I_{t-s} \, w_s$$

**Set `mean_GI` and `var_GI` below for your own disease** - the values here are a measles example (~11 day mean generation interval), not a default assumption.


In [ ]:
mean_GI <- 11/7    # EXAMPLE (measles): ~11 days, converted to weeks
var_GI  <- (4/7)^2  # EXAMPLE: SD ~4 days, converted to weeks and squared to variance

Lam <- computeLambda(I, mean_GI, var_GI)

plot(timevect, Lam, type = "l", col = "darkgreen", lwd = 2,
     xlab = "Time", ylab = expression(Lambda[t]), main = "Total infectiousness")
points(timevect, Lam, pch = 19, col = "darkgreen")

**Interpretation:** $\Lambda_t$ tracks the "pool" of currently-infectious potential, weighted by how likely each past case is to still be generating new infections right now. It rises and falls with a short lag behind the raw case counts - this lag is exactly what the generation interval encodes, and it's the denominator that $R_t$ estimation divides transmission by at each time point.

## Step 3: Bayesian filtering (real-time $R_t$)

EpiFilter estimates $R_t$ using a Bayesian recursive filter: a **prediction step** (propagate yesterday's belief about $R$ forward, allowing for some random drift) followed by an **update step** (correct that belief using today's actual case count, via the Poisson renewal likelihood):

$$I_t \sim \text{Poisson}(R_t \, \Lambda_t)$$

Critically, this is **causal** - the estimate at time $t$ only uses data up through time $t$, exactly what would have been known in real time on that day. This is the estimate a health department would actually have had available at each point in the outbreak.


In [ ]:
Rmin <- 0.01; Rmax <- 10; eta <- 0.1   # eta = state noise / smoothness of the R_t random walk
m <- 200
pR0 <- (1/m) * rep(1, m)
Rgrid <- seq(Rmin, Rmax, length.out = m)
a <- 0.025  # -> 95% credible intervals

tvec <- 2:n  # Lambda[1] is undefined (no prior incidence to convolve with)

Rfilt <- epiFilter(Rgrid, m, eta, pR0, length(tvec), Lam[tvec], I[tvec], a)
# Rfilt: [Rmed, Rhatci, Rmean, pR, pRup, pstate]

plot(timevect[tvec], Rfilt[[3]], type = "l", col = "blue", lwd = 2,
     ylim = c(0, max(Rfilt[[2]][4,])),
     xlab = "Time", ylab = expression(R[t]), main = "Filtered (real-time) R_t")
lines(timevect[tvec], Rfilt[[2]][1,], col = "blue", lty = 2)
lines(timevect[tvec], Rfilt[[2]][4,], col = "blue", lty = 2)
abline(h = 1, lty = 3)

cat("Most recent (current) R_t estimate:", round(tail(Rfilt[[3]],1), 3),
    "(95% CI:", round(tail(Rfilt[[2]][1,],1),3), "-", round(tail(Rfilt[[2]][4,],1),3), ")\n")

**Interpretation:** Watch for where this crosses $R_t = 1$ (dotted line) - that's the mathematically precise point where the outbreak's trajectory switches from growing to shrinking. Notice the credible interval is wide early on (little data to estimate from yet) and narrows as more cases accumulate - the model is honestly reporting its own uncertainty rather than pretending to know more than the data supports.

## Step 4: Smoothing (retrospective $R_t$)

The filtered estimate above only looks backward in time. **Smoothing** reruns the estimate using the *entire* dataset - past and future relative to each time point - giving the most accurate possible retrospective picture, at the cost of no longer being something you could have known in real time.

Comparing filtered vs. smoothed is genuinely informative: it shows you the difference between "what we knew then" and "what we know now, with hindsight."


In [ ]:
Rsmooth <- epiSmoother(Rgrid, m, Rfilt[[4]], Rfilt[[5]], length(tvec), Rfilt[[6]], a)
# Rsmooth: [Rmed, Rhatci, Rmean, qR]

plot(timevect[tvec], Rfilt[[3]], type = "l", col = "blue", lwd = 2,
     ylim = c(0, max(Rsmooth[[2]][4,], Rfilt[[2]][4,])),
     xlab = "Time", ylab = expression(R[t]), main = "Filtered vs Smoothed R_t")
lines(timevect[tvec], Rfilt[[2]][1,], col = "blue", lty = 2)
lines(timevect[tvec], Rfilt[[2]][4,], col = "blue", lty = 2)
lines(timevect[tvec], Rsmooth[[3]], col = "red", lwd = 2)
lines(timevect[tvec], Rsmooth[[2]][1,], col = "red", lty = 2)
lines(timevect[tvec], Rsmooth[[2]][4,], col = "red", lty = 2)
abline(h = 1, lty = 3)
legend("topright", c("Filtered (real-time)", "Smoothed (retrospective)", "R=1"),
       col = c("blue", "red", "black"), lty = c(1,1,3))

**Interpretation:** The two lines typically agree well once enough data has accumulated on both sides of a given time point, but can diverge noticeably early in the series or right at the most recent data point - exactly where "real-time" knowledge is most limited. If your filtered and smoothed estimates disagree a lot at the end of your series, that's a signal the most recent real-time estimate should be treated with extra caution.

## Step 5: One-step-ahead predictions (model adequacy check)

If the $R_t$ estimates are good, they should let us predict *next* period's case count reasonably well, using only information available up to the *previous* period:

$$I_{t} \mid R_{t-1} \sim \text{Poisson}(R_{t-1} \Lambda_t)$$

Comparing these predictions against what actually happened is a genuine model-adequacy check - not a retrospective fit, since the model never sees period $t$'s own outcome before predicting it.


In [ ]:
maxI <- max(800, ceiling(max(I, na.rm = TRUE) * 3))  # prediction grid, sized to your data
Ifilt <- recursPredictExt(Rgrid, Rfilt[[4]], Lam[tvec], Rfilt[[3]], a, maxI = maxI)
# Ifilt: [pred, predci]

tvec_pred <- tvec[-1]  # first filtered time point has no prior R to predict from

plot(timevect[tvec_pred], I[tvec_pred], type = "n",
     ylim = c(0, max(c(I[tvec_pred], Ifilt[[2]][2,]))*1.1),
     xlab = "Time", ylab = "Cases", main = "Observed vs one-step-ahead predicted")
xx <- timevect[tvec_pred]
polygon(c(xx, rev(xx)), c(Ifilt[[2]][1,], rev(Ifilt[[2]][2,])), col = rgb(0,0,1,0.15), border = NA)
lines(xx, Ifilt[[1]], col = "blue", lwd = 2)
points(xx, I[tvec_pred], pch = 19, col = "black")
legend("topright", c("Observed", "Predicted (in-sample)", "95% CI"),
       col = c("black","blue", rgb(0,0,1,0.4)), pch = c(19,NA,15), lty = c(NA,1,NA))

**Interpretation:** Most observed points should fall inside the shaded 95% interval if the model is well-specified. Occasional misses are expected (that's what a 95% interval means - roughly 1 in 20 points may fall outside by chance); a *pattern* of consistent misses (e.g. observed values always above the predicted band during the growth phase) would suggest the generation interval or model assumptions need revisiting.

## Step 6: Out-of-sample forecast

Everything above is retrospective (or in-sample). Now we genuinely forecast **one step beyond your last observed data point**, using only the current $R_t$ estimate and data through the present - nothing here has been "peeked at."


In [ ]:
I_extended <- c(I, NA)
Lam_next <- computeLambda(I_extended, mean_GI, var_GI)
Lam_forecast <- tail(Lam_next, 1)

pR_last <- Rfilt[[4]][nrow(Rfilt[[4]]), ]  # current posterior over R

Igrid <- 0:maxI
rate <- Lam_forecast * Rgrid
pI <- sapply(Igrid, function(k) sum(dpois(k, rate) * pR_last))
Fpred <- cumsum(pI) / sum(pI)

forecast_mean <- Lam_forecast * Rfilt[[3]][length(Rfilt[[3]])]
forecast_lo95 <- Igrid[which(Fpred >= a)[1]]
forecast_hi95 <- Igrid[which(Fpred >= 1-a)[1]]
forecast_time <- timevect[n] + 1

cat("Forecast for time", forecast_time, "(no observed data yet):\n")
cat(sprintf("  Predicted cases: %.1f (95%% CI: %d-%d)\n", forecast_mean, forecast_lo95, forecast_hi95))

plot(xx, I[tvec_pred], type = "n",
     xlim = c(min(xx), forecast_time+0.5),
     ylim = c(0, max(c(I[tvec_pred], Ifilt[[2]][2,], forecast_hi95))*1.1),
     xlab = "Time", ylab = "Cases", main = "Current situation + forecast")
polygon(c(xx, rev(xx)), c(Ifilt[[2]][1,], rev(Ifilt[[2]][2,])), col = rgb(0,0,1,0.15), border = NA)
lines(xx, Ifilt[[1]], col = "blue", lwd = 2)
points(xx, I[tvec_pred], pch = 19, col = "black")
abline(v = timevect[n], lty = 3)
segments(timevect[n], I[n], forecast_time+0.5, I[n], col = "gray50", lty = 3)
points(forecast_time, forecast_mean, col = "red", pch = 19, cex = 1.3)
arrows(forecast_time, forecast_lo95, forecast_time, forecast_hi95, col="red", angle=90, code=3, length=0.05, lwd=2)
legend("topleft", c("Observed","Predicted (in-sample)","Forecast","Last observed value (ref.)"),
       col=c("black","blue","red","gray50"), pch=c(19,NA,19,NA), lty=c(NA,1,NA,3), cex=0.8)

**Interpretation:** The gray dotted reference line marks the last observed value - if the red forecast point sits above it, the model expects cases to rise next period; below, a continued decline. This single-point forecast should be treated as provisional: it reflects only the current $R_t$ estimate and assumes no sudden change in transmission. As each new period's data arrives, re-running this notebook rolls the forecast forward by one step, and you can check this forecast against what actually happened - genuine forecast-performance tracking over time.

## Step 7: Elimination probability

A directly decision-relevant summary: the probability that $R_t < 1$ at each time point, computed from the full posterior distribution over $R$ (not just its mean).


In [ ]:
below1 <- Rgrid < 1
Pelim <- apply(Rfilt[[4]], 1, function(p) sum(p[below1]))

plot(timevect[tvec], Pelim, type = "l", lwd = 2, col = "purple",
     ylim = c(0,1), xlab = "Time", ylab = expression(P(R[t] < 1)),
     main = "Elimination probability over time")
abline(h = 0.5, lty = 3)

cat("Current P(R_t < 1):", round(tail(Pelim,1), 3), "\n")

**Interpretation:** This crosses 50% at roughly the same point $R_t$ crosses 1, as it should - but it also tells you *how confident* that crossing is. A value near 99% is a strong, low-uncertainty signal that transmission has genuinely fallen below replacement; a value hovering near 50% means the data doesn't yet clearly support either conclusion, which is itself useful information for a decision-maker weighing whether to declare an outbreak controlled.

## Next steps

- Full repository, with the batch-organized output structure and CSV exports for tracking forecast performance over time: [github.com/rajsubediresearch/RtForecastR](https://github.com/rajsubediresearch/RtForecastR)
- To use this for your own outbreak: replace the data in Step 1, and set `mean_GI`/`var_GI` in Step 2 to a properly-sourced generation interval for your specific disease.
- This method is disease-agnostic - nothing here is measles-specific except the example data and generation interval values.
